In [2]:
import pandas as pd
import numpy as np

In [3]:
COLUMNS = [
    "id", "gender", "masterCategory", "subCategory", "articleType",
    "baseColour", "season", "year", "usage", "productDisplayName",
]
def read_styles(path):
    rows = []
    with open(path, encoding="utf-8") as fh:
        next(fh)  # skip header
        for line in fh:
            fields = line.rstrip("\n").split(",", 9)  # max 10 parts
            if len(fields) < len(COLUMNS):
                fields += [""] * (len(COLUMNS) - len(fields))
            rows.append(fields)
    return pd.DataFrame(rows, columns=COLUMNS)

df = read_styles("styles.csv")
images_link=pd.read_csv("images.csv")

In [4]:
df

,id,gender,masterCategory,subCategory,articleType,baseColour,season,year,usage,productDisplayName
0,15970,Men,Apparel,Topwear,Shirts,Navy Blue,Fall,2011,Casual,Turtle Check Men Navy Blue Shirt
1,39386,Men,Apparel,Bottomwear,Jeans,Blue,Summer,2012,Casual,Peter England Men Party Blue Jeans
2,59263,Women,Accessories,Watches,Watches,Silver,Winter,2016,Casual,Titan Women Silver Watch
3,21379,Men,Apparel,Bottomwear,Track Pants,Black,Fall,2011,Casual,Manchester United Men Solid Black Track Pants
4,53759,Men,Apparel,Topwear,Tshirts,Grey,Summer,2012,Casual,Puma Men Grey T-shirt
...,...,...,...,...,...,...,...,...,...,...
44441,17036,Men,Footwear,Shoes,Casual Shoes,White,Summer,2013,Casual,Gas Men Caddy Casual Shoe
44442,6461,Men,Footwear,Flip Flops,Flip Flops,Red,Summer,2011,Casual,Lotto Men's Soccer Track Flip Flop
44443,18842,Men,Apparel,Topwear,Tshirts,Blue,Fall,2011,Casual,Puma Men Graphic Stellar Blue Tshirt
44444,46694,Women,Personal Care,Fragrance,Perfume and Body Mist,Blue,Spring,2017,Casual,Rasasi Women Blue Lady Perfume


In [5]:
df.drop(columns=['year'],inplace=True)
df

,id,gender,masterCategory,subCategory,articleType,baseColour,season,usage,productDisplayName
0,15970,Men,Apparel,Topwear,Shirts,Navy Blue,Fall,Casual,Turtle Check Men Navy Blue Shirt
1,39386,Men,Apparel,Bottomwear,Jeans,Blue,Summer,Casual,Peter England Men Party Blue Jeans
2,59263,Women,Accessories,Watches,Watches,Silver,Winter,Casual,Titan Women Silver Watch
3,21379,Men,Apparel,Bottomwear,Track Pants,Black,Fall,Casual,Manchester United Men Solid Black Track Pants
4,53759,Men,Apparel,Topwear,Tshirts,Grey,Summer,Casual,Puma Men Grey T-shirt
...,...,...,...,...,...,...,...,...,...
44441,17036,Men,Footwear,Shoes,Casual Shoes,White,Summer,Casual,Gas Men Caddy Casual Shoe
44442,6461,Men,Footwear,Flip Flops,Flip Flops,Red,Summer,Casual,Lotto Men's Soccer Track Flip Flop
44443,18842,Men,Apparel,Topwear,Tshirts,Blue,Fall,Casual,Puma Men Graphic Stellar Blue Tshirt
44444,46694,Women,Personal Care,Fragrance,Perfume and Body Mist,Blue,Spring,Casual,Rasasi Women Blue Lady Perfume


In [6]:
cols = [
    "gender", "masterCategory", "subCategory", "articleType",
    "baseColour", "season", "usage", "productDisplayName",
]
df["product_search_description"] = (
    df[cols]
    .fillna("")
    .agg(" ".join, axis=1)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
    .str.lower()
)
df = df[["id", "product_search_description"]]


In [7]:
df

,id,product_search_description
0,15970,men apparel topwear shirts navy blue fall casu...
1,39386,men apparel bottomwear jeans blue summer casua...
2,59263,women accessories watches watches silver winte...
3,21379,men apparel bottomwear track pants black fall ...
4,53759,men apparel topwear tshirts grey summer casual...
...,...,...
44441,17036,men footwear shoes casual shoes white summer c...
44442,6461,men footwear flip flops flip flops red summer ...
44443,18842,men apparel topwear tshirts blue fall casual p...
44444,46694,women personal care fragrance perfume and body...


In [8]:
import json

In [15]:
from pathlib import Path
def lad_json_file(id:str):
    file_path=Path(f"./styles/{id}.json")

    with open(file_path,"r",encoding="utf-8") as f:
        data=json.load(f)
    return data

In [10]:
def filter_product(raw):
    payload = raw.get("data", {})
    return {
        "id": payload.get("id"),
        "name": payload.get("productDisplayName"),
        "variant": payload.get("variantName"),
        "brand": payload.get("brandName"),
        "price": payload.get("price"),
        "discounted_price": payload.get("discountedPrice"),
        "usage": payload.get("usage"),
        "image_url": payload.get("styleImages", {})
                          .get("default", {})
                          .get("imageURL"),
    }


In [11]:
import os

In [16]:
df['meta_data']=(df['id'].apply(lad_json_file)).apply(filter_product)

In [17]:
df

,id,product_search_description,meta_data
0,15970,men apparel topwear shirts navy blue fall casu...,"{'id': 15970, 'name': 'Turtle Check Men Navy B..."
1,39386,men apparel bottomwear jeans blue summer casua...,"{'id': 39386, 'name': 'Peter England Men Party..."
2,59263,women accessories watches watches silver winte...,"{'id': 59263, 'name': 'Titan Women Silver Watc..."
3,21379,men apparel bottomwear track pants black fall ...,"{'id': 21379, 'name': 'Manchester United Men S..."
4,53759,men apparel topwear tshirts grey summer casual...,"{'id': 53759, 'name': 'Puma Men Grey T-shirt',..."
...,...,...,...
44441,17036,men footwear shoes casual shoes white summer c...,"{'id': 17036, 'name': 'Gas Men Caddy Casual Sh..."
44442,6461,men footwear flip flops flip flops red summer ...,"{'id': 6461, 'name': 'Lotto Men's Soccer Track..."
44443,18842,men apparel topwear tshirts blue fall casual p...,"{'id': 18842, 'name': 'Puma Men Graphic Stella..."
44444,46694,women personal care fragrance perfume and body...,"{'id': 46694, 'name': 'Rasasi Women Blue Lady ..."


In [18]:
df.to_csv("final_filtered_data.csv",index=False)